# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Raw table grain:** one row = one (`report_date`, `client_hash_id`, `content_hash_id`) — a single content item's performance on a single day, in `fact_content_daily_performance`.

**My lane's analysis grain:** one row = one (`client_hash_id`, `content_hash_id`) content item, aggregated over a single mid-panel month. I use **March 2026** (`month=2026-03`) — a middle month, not the sealed final month (June 2026) and not the `_sample` table (which *is* the final month and would leak the future into anything I try to predict).

**The window is split in two, on purpose:**
- **Feature window:** March 1–15 — everything a reviewer would already know by the "decision moment" of mid-March.
- **Target window:** March 16–31 — what actually happened next, used only to build the label, never as a feature.

This mirrors the prior-window → future-window shape the lane guide recommends, just compressed into one month instead of the full 90/30-day split, since Week 3 only asks for one month of warehouse data.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Context** (grouping/joining only, never model inputs): `client_hash_id`, `content_hash_id`.

**Feature** (from the March 1–15 window only — knowable before the decision moment): `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`.

**Label / proxy:** `is_declining_label` — 1 if a content item's March 16–31 impressions are lower than its March 1–15 impressions, else 0. Built only from the target window, never fed back in as a feature.

**Excluded, each with a reason:**
- `sessions_ai` and every `ai_*` column — Week 1's own numbers already showed AI-referral rows are far sparser than search impressions (30,177 vs 28.9M in the starter slice). Not enough density in one month to be a reliable refresh-scoring signal; the lane guide explicitly warns against building on sparse AI-session data.
- Every March 16–31 metric as a *feature* — this is the exact quantity the label is built from. Using it as an input would be circular. Part 3 below deliberately breaks this rule once, on purpose, to show what it does to the score.

In [2]:
CONTEXT_COLS = ["client_hash_id", "content_hash_id"]
FEATURE_COLS = ["impressions_h1", "clicks_h1", "avg_position_h1", "sessions_h1", "engaged_sessions_h1"]
LABEL_COL = "is_declining_label"
EXCLUDED = {
    "sessions_ai / ai_*": "too sparse in one month to be a reliable signal (Week 1: 30,177 vs 28.9M rows)",
    "any March 16-31 metric as a feature": "defines the label -- using it as an input would be circular (the trap in Part 3)",
}

print("context:", CONTEXT_COLS)
print("features (5):", FEATURE_COLS)
print("label:", LABEL_COL)
print("excluded:")
for k, v in EXCLUDED.items():
    print(f"  - {k}: {v}")

context: ['client_hash_id', 'content_hash_id']
features (5): ['impressions_h1', 'clicks_h1', 'avg_position_h1', 'sessions_h1', 'engaged_sessions_h1']
label: is_declining_label
excluded:
  - sessions_ai / ai_*: too sparse in one month to be a reliable signal (Week 1: 30,177 vs 28.9M rows)
  - any March 16-31 metric as a feature: defines the label -- using it as an input would be circular (the trap in Part 3)


## 3. Verify it with queries (grain, counts, availability) — then five features, then the trap

Three claims from Parts 1-2, each backed by a real query against `hf://datasets/FlyRank/internship-warehouse`, month partition `2026-03`. Token comes from the `HF_TOKEN` environment variable — never hardcoded, never printed.

In [3]:
import os
import duckdb
import pandas as pd

token = os.environ["HF_TOKEN"]  # set on your own machine; never pasted into this cell
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

MONTH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# --- Query 1: grain -- one row really is one (date, client, content) ---
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MONTH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Query 1 -- grain check (rows with duplicate date+client+content; empty = grain holds):")
print(grain_check if len(grain_check) else "  (empty -- grain confirmed)")

Query 1 -- grain check (rows with duplicate date+client+content; empty = grain holds):
  (empty -- grain confirmed)


In [4]:
# --- Query 2: row count + date span for the month partition ---
span_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {MONTH}
""").df()

print("Query 2 -- row count + date span for month=2026-03:")
print(span_check.to_string(index=False))

Query 2 -- row count + date span for month=2026-03:
 n_rows   min_date   max_date  n_clients  n_content
9841378 2026-03-01 2026-03-31         55     331437


In [5]:
# --- Query 3: availability, filtered with IS TRUE -- how many rows survive? ---
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {MONTH}
""").df()

total = int(availability.loc[0, "total_rows"])
gsc_ok = int(availability.loc[0, "gsc_available_rows"])
ga4_ok = int(availability.loc[0, "ga4_available_rows"])

print("Query 3 -- availability (IS TRUE filter):")
print(f"  total rows:              {total:,}")
print(f"  gsc_data_available=TRUE: {gsc_ok:,}  ({gsc_ok/total*100:.1f}% of rows survive)")
print(f"  ga4_data_available=TRUE: {ga4_ok:,}  ({ga4_ok/total*100:.1f}% of rows survive)")

Query 3 -- availability (IS TRUE filter):
  total rows:              9,841,378
  gsc_data_available=TRUE: 3,611,061  (36.7% of rows survive)
  ga4_data_available=TRUE: 413,966  (4.2% of rows survive)


### Five features (max), built from the March 1–15 window only

Each one, with its "knowable at the decision moment because…" line:

1. **`impressions_h1`** — knowable because it only sums `gsc_impressions` for dates `<= 2026-03-15`, before the decision point.
2. **`clicks_h1`** — same window restriction; observed search clicks through March 15.
3. **`avg_position_h1`** — average GSC position through March 15 only; today's rank, not a future one.
4. **`sessions_h1`** — GA4 sessions through March 15; an already-happened engagement count.
5. **`engaged_sessions_h1`** — engaged-session subset through March 15; same window discipline.

All five are aggregated remotely in SQL (never a raw 9.8M-row pull into pandas) and joined at the (`client_hash_id`, `content_hash_id`) grain confirmed in Query 1.

In [6]:
# Feature window: March 1-15. Aggregate remotely, bring back only the small result.
features_h1 = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_h1,
        SUM(gsc_clicks) AS clicks_h1,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_h1,
        SUM(ga4_sessions) AS sessions_h1,
        SUM(ga4_engaged_sessions) AS engaged_sessions_h1
    FROM {MONTH}
    WHERE report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# Target window: March 16-31. Used ONLY to build the label -- never joined in as a feature.
target_h2 = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_h2
    FROM {MONTH}
    WHERE report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"features_h1: {features_h1.shape[0]:,} content items")
print(f"target_h2:   {target_h2.shape[0]:,} content items")
features_h1.head()

features_h1: 151,981 content items
target_h2:   166,224 content items


,client_hash_id,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,sessions_h1,engaged_sessions_h1
0,client_400c21c81c8b46ef,content_5724abbe21dcbc9e,2.0,0.0,8.500000,0.0,0.0
1,client_400c21c81c8b46ef,content_1bc67e9d435b3a78,264.0,0.0,6.176354,0.0,0.0
2,client_400c21c81c8b46ef,content_09a57239b2297f55,39.0,0.0,5.422176,0.0,0.0
3,client_400c21c81c8b46ef,content_8a598c75629f8791,9.0,0.0,7.666667,0.0,0.0
4,client_400c21c81c8b46ef,content_81dfd8f7f5b040fc,106.0,0.0,5.044288,0.0,0.0


### The trap

Build the label honestly first (front-half feature volume filter, so the label isn't noise on near-zero-impression pages). Then: add ONE column derived straight from the target window as if it were a feature, watch a quick logistic-regression score jump toward perfect, delete it, and keep only the honest number.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Join features to target, build the label. Minimum front-half volume so the label
# isn't just noise on pages with a handful of impressions.
data = features_h1.merge(target_h2, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["impressions_h1"] >= 50].copy()
data["is_declining_label"] = (data["impressions_h2"] < data["impressions_h1"]).astype(int)
data = data.fillna(0)

print(f"Labeled rows (impressions_h1 >= 50): {len(data):,}")
print("Label balance:")
print(data["is_declining_label"].value_counts(normalize=True).round(3))

X_honest = data[FEATURE_COLS]
y = data["is_declining_label"]
Xh_tr, Xh_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=0, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(Xh_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(Xh_te)[:, 1])
print(f"\nHonest AUC (5 front-half features only): {honest_auc:.3f}")

# --- Now spring the trap on purpose: add a column derived from the TARGET window ---
data["LEAK_impressions_h2"] = data["impressions_h2"]  # this literally defines the label
X_leaky = data[FEATURE_COLS + ["LEAK_impressions_h2"]]
Xl_tr, Xl_te, _, _ = train_test_split(X_leaky, y, test_size=0.3, random_state=0, stratify=y)

leaky_model = LogisticRegression(max_iter=1000).fit(Xl_tr, y_tr)
leaky_auc = roc_auc_score(y_te, leaky_model.predict_proba(Xl_te)[:, 1])
print(f"LEAKY AUC (same 5 features + the target-window column): {leaky_auc:.3f}  <-- jumps toward 1.0")

# Delete the leaked column. Keep only the honest number.
data = data.drop(columns=["LEAK_impressions_h2"])
print(f"\nLeaked column removed. Kept, honest result: AUC = {honest_auc:.3f} (not {leaky_auc:.3f}).")

Labeled rows (impressions_h1 >= 50): 92,247
Label balance:
is_declining_label
0    0.563
1    0.437
Name: proportion, dtype: float64



Honest AUC (5 front-half features only): 0.599


LEAKY AUC (same 5 features + the target-window column): 1.000  <-- jumps toward 1.0

Leaked column removed. Kept, honest result: AUC = 0.599 (not 1.000).


## 4. Data limits

**Named limitation: unbalanced client history within the month.** Not every client necessarily has tracking covering the full March 1–31 window — some clients' GSC/GA4 history starts partway through the panel (per `dim_clients.gsc_data_start` / `ga4_data_start`). A client whose tracking started March 20th would show near-zero front-half volume for reasons that have nothing to do with content performance, and my `impressions_h1 >= 50` filter only partly protects against that — it drops thin rows but doesn't distinguish "genuinely low-traffic page" from "page whose tracking just hadn't started yet." Checked below with a real query, not asserted.

**Second limitation, smaller but real: the label window is short.** A 15-day vs. 15-day within-month comparison is noisier than the starter's 90-day-window proxy, which is itself already flagged (Week 2) as weaker than a true future-outcome label. This month-level version is a rougher proxy still — fine for this week's exercise, not something I'd ship a real recommendation from without a longer, more careful window.

**What this data can never tell you (structural, not just this month):** whether a refresh *caused* any later recovery — that needs an experiment. And nothing here reveals Google's actual ranking algorithm; only observed search/engagement outcomes.

In [8]:
# Supporting check for the "unbalanced client history" limitation:
# how many distinct days of March does each client actually have data for?
client_coverage = con.sql(f"""
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS days_covered
    FROM {MONTH}
    GROUP BY client_hash_id
    ORDER BY days_covered ASC
""").df()

full_month = (client_coverage["days_covered"] == 31).sum()
partial = (client_coverage["days_covered"] < 31).sum()
print(f"Clients with all 31 days of March: {full_month}")
print(f"Clients with PARTIAL March coverage: {partial}")
print("\nThinnest coverage (bottom 5 clients by days_covered):")
print(client_coverage.head(5).to_string(index=False))

Clients with all 31 days of March: 51
Clients with PARTIAL March coverage: 4

Thinnest coverage (bottom 5 clients by days_covered):
         client_hash_id  days_covered
client_e00b29e582949543             9
client_810019792c9b8efc            12
client_f6f0cdf26d03d7bd            13
client_86ebc2f12c01f586            29
client_ff644d8251367cbb            31


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.